In [1]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import pandas as pd

In [2]:
def recode_nssec(df):

    df = df.copy()

    df.loc[df["NSSEC5"].isna() & (df["Age9"] >= 8), "NSSEC5"] = 5

    return df

In [3]:
def recode_workstat(df):
    df = df.copy()

    mapping = {1: 1,
               2: 2,
               3: 3,
               4: 3,
               5: 4,
               6: 5,
               7: 6,
               8: 7,
               9: 7,
               10: 8}

    df["WorkStat8"] = df["WorkStat10"].map(mapping)

    return df

In [4]:
def recode_hhliv(df):
    df = df.copy()

    mapping = {1: 1,
               2: 2,
               3: 3,
               4: 3,
               5: 4,
               6: 5,
               7: 6,
               8: 7,
               9: 7,
               10: 8,
               11: 9,
               12: 9}

    df["HHLiv9"] = df["HHLiv12"].map(mapping)

    return df

In [6]:
overall_df = pd.read_csv("../data/master_data/2016_to_2023_full_preprocessed_data_set.csv.gz")

In [6]:
motivation_cols = ["Motiva_POP", "motivb_POP", "motivc_POP"]

motivation_df = overall_df[motivation_cols].dropna().copy()

scaler = StandardScaler()
motivation_scaled = scaler.fit_transform(motivation_df)

pca = PCA(n_components=1)
pc1 = pca.fit_transform(motivation_scaled).flatten()

print(f"PC1 explains {pca.explained_variance_ratio_[0]:.3f} of the variance")

motivation_df["Motivation_PC"] = pc1

overall_df = overall_df.merge(motivation_df[["Motivation_PC"]], left_index=True, right_index=True, how="left")

overall_df = overall_df.copy()

overall_df["Motivation_PC_Q"] = pd.qcut(overall_df["Motivation_PC"], q=5, labels=False)

PC1 explains 0.666 of the variance


In [7]:
loadings = pd.DataFrame(pca.components_.T, index=motivation_cols, columns=["Loading"])

print(loadings)

             Loading
Motiva_POP  0.588287
motivb_POP  0.625731
motivc_POP  0.512229


In [8]:
overall_df = recode_nssec(overall_df)
overall_df = recode_workstat(overall_df)
overall_df = recode_hhliv(overall_df)

In [9]:
cluster_cols = ["serial",
                "year",
                "Age9",
                "Gend3",
                "Eth7",
                "Disab2_POP",
                "Educ6",
                "NSSEC5",
                "IMD10",
                "WorkStat8",
                "Child4",
                "HHLiv9",
                "Motivation_PC_Q",
                "motivd_POP"]

In [10]:
cluster_df = overall_df[cluster_cols].copy()
cluster_df = cluster_df.dropna().copy()

In [11]:
cluster_vars = [col for col in cluster_cols if col not in ["serial", "year"]]

cluster_df[cluster_vars] = cluster_df[cluster_vars].astype(int)

for col in cluster_vars:
    cluster_df[col] -= cluster_df[col].min()

In [12]:
cluster_df.to_csv("2016_to_2023_clustering_data_set.csv", index=False)